In [ ]:
#这周设置的攻击，忘记加入PGD了所以只是针对mse施加的随机噪声，效果不是很好
from google.colab import drive
drive.mount('/content/drive')

#重建ae_gcn结构，加载det_loss微调后的权重
ae_gcn = build_gcn_ae(ae_attn, GCNSkipAEConfig())
ae_gcn.load_state_dict(torch.load(
    '/content/drive/MyDrive/gcn_ae_checkpoints/gcn_ae_ckpt_det.pt',
    map_location=bld.DEVICE
))

#加载diffusion模型
betas, flip_prob = bld.get_schedule()
diff_model = bld.ReverseModel(D=ae_gcn.D).to(bld.DEVICE)
diff_model.load_state_dict(torch.load(
    '/content/drive/MyDrive/gcn_ae_checkpoints/gcn_diffusion.pt', map_location=bld.DEVICE
))

#detector + 全量测试集
from ultralytics import YOLO
detector = YOLO('/content/drive/MyDrive/dfire_checkpoints/yolov8n_dfire_detector.pt')
test_dataset = bld.DFireDataset(bld.DFIRE_ROOT, split="test")

#全量评估
results = evaluate_purification_map(
    diff_model, ae_gcn, test_dataset, detector, betas, flip_prob,
    flip_rates=[0.1], t_stars=[10, 40],
    n_samples=None,   # 全量4306张
    save_dir_root='/content/eval_purify_map_full',
)

In [ ]:
#原来设置的pgd攻击函数，在下一个code块里面被调用
def pgd_attack(model, criterion, x, targets, eps=8/255, alpha=2/255, steps=10):
    """
    model   : detector.model (DetectionModel)
    x       : (B,3,H,W) float [0,1]，注意要 resize 到训练时的 imgsz=384
    targets : dict，需包含 'cls', 'bboxes', 'batch_idx' 三个key（YOLO格式）
    """
    x_adv = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(0, 1).requires_grad_(True)
    for _ in range(steps):
        preds = model(x_adv)
        loss, _ = criterion(preds, targets)
        loss = loss.sum()
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = (x_adv.detach() + alpha * grad.sign()).clamp(x - eps, x + eps).clamp(0, 1)
        x_adv.requires_grad_(True)
    return x_adv.detach()

In [ ]:
#原先版本的PGD攻击
#这个版本里clean的mAP50有0.556，攻击后的mAP50下降到了0.0756，净化后的mAP50回升到0.1726，净化效率在20%左右，比注意力机制加入的最优版本0.1814略低
import os
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ---------------- 彻底修复 criterion.hyp 的兼容性包装类 ----------------
class DictAttributeWrapper(dict):
    """同时支持 dict['key'] 和 dict.key 访问的字典包装类"""
    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(f"'DictAttributeWrapper' object has no attribute '{name}'")

    def __setattr__(self, name, value):
        self[name] = value

    def __delattr__(self, name):
        try:
            del self[name]
        except KeyError:
            raise AttributeError(f"'DictAttributeWrapper' object has no attribute '{name}'")

# 应用包装类修复 criterion.hyp
if hasattr(criterion, 'hyp') and isinstance(criterion.hyp, dict):
    criterion.hyp = DictAttributeWrapper(criterion.hyp)

# ==============================================================================
#  1. 初始化与权重加载
# ==============================================================================
levels = auto_detect_skip_levels(ae_attn, img_size=bld.IMG_SIZE)
ae_gcn_config = GCNSkipAEConfig()

ae_gcn = BinaryAutoEncoderGCN(ae_attn, ae_gcn_config, levels).to(bld.DEVICE)
ae_gcn.load_state_dict(
    torch.load('/content/drive/MyDrive/gcn_ae_checkpoints/gcn_ae_ckpt.pt', map_location=bld.DEVICE)
)
ae_gcn.eval()
print("gcn_ae 权重加载完成")

diff_model_v3 = bld.ReverseModel(D=ae_gcn.D).to(bld.DEVICE)
diff_model_v3_path = '/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion_v3.pt'
diff_model_v3.load_state_dict(torch.load(diff_model_v3_path, map_location=bld.DEVICE))
diff_model_v3.eval()
print("diff_model_v3 权重加载完成")

T_STAR = 40
MODE = "realistic"
OUT_ROOT = f'/content/drive/MyDrive/dfire_adv_eval_gcn_pgd_{MODE}'

for split in ['clean', 'adv', 'purified']:
    os.makedirs(f'{OUT_ROOT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUT_ROOT}/{split}/labels', exist_ok=True)

progress_path = f'{OUT_ROOT}/done.txt'
done = set(open(progress_path).read().split()) if os.path.exists(progress_path) else set()
print(f"已完成 {len(done)} 张图")
progress_f = open(progress_path, 'a')

betas, flip_prob = bld.get_schedule()

# ==============================================================================
#  2. 主循环：PGD攻击 -> GCN Skip 净化 -> 图像落盘
# ==============================================================================
for x, targets, paths in tqdm(det_loader, desc=f"gcn_pgd attack+purify [{MODE}]"):
    names = [os.path.basename(p) for p in paths]
    if all(n in done for n in names):
        continue

    x = x.to(bld.DEVICE)

    if targets['bboxes'].shape[0] == 0:
        x_adv = x.clone()
    else:
        targets_gpu = {k: v.to(bld.DEVICE) for k, v in targets.items()}
        x_adv = pgd_attack(attack_yolo.model, criterion, x, targets_gpu)

    x_adv_128   = F.interpolate(x_adv, size=bld.IMG_SIZE, mode='bilinear', align_corners=False)
    x_clean_128 = F.interpolate(x,     size=bld.IMG_SIZE, mode='bilinear', align_corners=False)

    with torch.no_grad():
        z_adv, feats_adv = ae_gcn.encoder(x_adv_128, hard=True)
        z_adv_flat = z_adv.view(z_adv.shape[0], -1).long()

        t_vec = torch.full((z_adv_flat.shape[0],), T_STAR - 1, device=bld.DEVICE, dtype=torch.long)
        
        z_noised = bld.q_sample(z_adv_flat, t_vec, flip_prob)
        z_rec = bld.reverse(diff_model_v3, z_noised, T_STAR, betas, flip_prob).float()
        z_rec = z_rec.view(z_adv.shape)

        if MODE == "realistic":
            feats_for_decode = feats_adv
        elif MODE == "oracle":
            _, feats_for_decode = ae_gcn.encoder(x_clean_128, hard=True)
        else:
            raise ValueError(f"未知 MODE: {MODE}")

        x_pur_128 = ae_gcn.decoder(z_rec, feats_for_decode)
    
    x_pur = F.interpolate(x_pur_128, size=384, mode='bilinear', align_corners=False)

    save_as_dfire_split(x,     paths, f'{OUT_ROOT}/clean')
    save_as_dfire_split(x_adv,  paths, f'{OUT_ROOT}/adv')
    save_as_dfire_split(x_pur,  paths, f'{OUT_ROOT}/purified')

    for n in names:
        progress_f.write(n + '\n')
    progress_f.flush()
    done.update(names)

progress_f.close()
print(f"\n全部完成，共 {len(done)} 张图")

# ==============================================================================
#  3. mAP 评估
# ==============================================================================
print(f"\n=== gcn_ae + 真实PGD攻击 [{MODE} mode] mAP对比 ===")
results = {}
for name in ['clean', 'adv', 'purified']:
    map50, map5095 = eval_map(f'{OUT_ROOT}/{name}', img_size=384)
    results[name] = (map50, map5095)
    print(f"{name:10s}  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")

recovery = (results['purified'][0] - results['adv'][0]) / max(results['clean'][0] - results['adv'][0], 1e-8)
print(f"\n净化恢复比例 [{MODE}] = {recovery:.1%}")